# QPEI Empirical Analysis Pipeline — Framework-Aligned
### Google Colab version | Master workbook → EFA/CFA → QPEI evidence

This notebook is designed specifically for the cleaned QPE master workbook.

**Important correction:** the actual workbook uses the sheets `Teacher_Survey`, `Parent_Survey`, and `Student_Questionnaire`, with item codes `tq1–tq30`, `pq1–pq20`, and `sq1–sq18`. The earlier notebook incorrectly looked for sheets named `Teacher`, `Parent`, `Student`, which is why the item lists were empty.

The analysis follows the established QPEI framework:

1. Teacher competence and pedagogical practice — **20%**
2. Curriculum implementation and assessment — **15%**
3. Learning environment and infrastructure — **15%**
4. Student learning outcomes and FLN — **20%**
5. Governance, management, and community support — **15%**
6. Equity and inclusiveness — **15%**

The methodology describes the QPEI as a **conceptual and diagnostic organising framework**, while the study's empirical findings retain the native **1–5 Likert/observation scales**. The 1–5 → 0–100 transformation is therefore kept as a QPEI scoring option rather than being imposed on every descriptive result. 

**EFA/CFA role:** identify and test empirical measurement structures within the respondent instruments. They do not automatically replace the six theoretical QPEI dimensions.


In [ ]:
# ============================================================
# 0. SETUP — GOOGLE COLAB
# ============================================================

!pip -q install factor_analyzer pingouin semopy openpyxl statsmodels scikit-learn seaborn

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, warnings, re, math
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
import pingouin as pg
from semopy import Model, calc_stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# -----------------------------
# YOUR CORRECTED DRIVE PATHS
# -----------------------------
DATA_PATH = Path("/content/drive/MyDrive/QPEI/QPE_MASTER_Cleaned_IDs_Translated_Analysis.xlsx")
RESULTS_PATH = Path("/content/drive/MyDrive/QPEI/results.json")
OUTPUT_DIR = Path("/content/drive/MyDrive/QPEI/qpei_outputs")

TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
LOG_DIR = OUTPUT_DIR / "logs"

for p in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("="*80)
print("QPEI ANALYSIS INITIALIZATION")
print("="*80)
print("Master:", DATA_PATH)
print("Master exists:", DATA_PATH.exists())
print("Results:", RESULTS_PATH)
print("Output:", OUTPUT_DIR)
print("Tables:", TABLE_DIR)
print("Figures:", FIGURE_DIR)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"MASTER WORKBOOK NOT FOUND:\n{DATA_PATH}\n\n"
        "Check MyDrive/QPEI/ and confirm the filename exactly."
    )

print("\n✓ Setup successful")


## 1. Load workbook and immediately create persistent outputs

Every major stage writes its table/figure to Drive **when that stage finishes**. Therefore, if a later EFA/CFA cell fails, the earlier outputs will still be present.

The notebook also displays the same outputs inline.


In [ ]:
# 1. LOAD MASTER WORKBOOK

xl = pd.ExcelFile(DATA_PATH)
print("Workbook:", DATA_PATH)
print("\nSheets found:")
for i, s in enumerate(xl.sheet_names, 1):
    print(f"  {i:02d}. {s}")

sheets = {s: pd.read_excel(DATA_PATH, sheet_name=s) for s in xl.sheet_names}

inventory = pd.DataFrame([
    {
        "sheet": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    }
    for name, df in sheets.items()
])

print("\nWORKBOOK INVENTORY")
display(inventory)

inventory.to_csv(TABLE_DIR / "Table_01_Workbook_Inventory.csv", index=False)
print("✓ Saved:", TABLE_DIR / "Table_01_Workbook_Inventory.csv")


In [ ]:
# 1A. PRINT COLUMN STRUCTURE — THIS IS THE FIRST DIAGNOSTIC

for name, df in sheets.items():
    print("\n" + "="*80)
    print(name, "→", df.shape)
    print("="*80)
    print(list(df.columns))


## 2. Exact instrument detection

We use the **actual sheet names and exact item-code prefixes** from the cleaned workbook.

- Teacher: `Teacher_Survey`, `tq1–tq30`
- Parent: `Parent_Survey`, `pq1–pq20`
- Student: `Student_Questionnaire`, `sq1–sq18`
- Classroom observation: `Classroom_Observation`, `co1–co28`
- School environment: `School_Environment`, `se1–se11`

Observation/environment data are retained for the school-level framework, but are **not subjected to ordinary respondent-level EFA** because their sample structures are different.


In [ ]:
# 2. DETECT ACTUAL ITEM LISTS

INSTRUMENTS = {
    "Teacher": {"sheet": "Teacher_Survey", "prefix": "tq"},
    "Parent": {"sheet": "Parent_Survey", "prefix": "pq"},
    "Student": {"sheet": "Student_Questionnaire", "prefix": "sq"},
    "Observation": {"sheet": "Classroom_Observation", "prefix": "co"},
    "Environment": {"sheet": "School_Environment", "prefix": "se"},
}

ITEMS = {}

for label, spec in INSTRUMENTS.items():
    df = sheets[spec["sheet"]]
    prefix = spec["prefix"]
    cols = [
        c for c in df.columns
        if re.fullmatch(fr"{prefix}\d+", str(c).strip().lower())
    ]
    cols = sorted(cols, key=lambda x: int(re.search(r"\d+", str(x)).group()))
    ITEMS[label] = cols

    print(f"{label:12s}: {len(cols):2d} items | {cols}")

# Backward-compatible aliases
TQ_ITEMS = ITEMS["Teacher"]
PQ_ITEMS = ITEMS["Parent"]
SQ_ITEMS = ITEMS["Student"]

assert len(TQ_ITEMS) == 30, f"Expected 30 teacher items, found {len(TQ_ITEMS)}"
assert len(PQ_ITEMS) == 20, f"Expected 20 parent items, found {len(PQ_ITEMS)}"
assert len(SQ_ITEMS) == 18, f"Expected 18 student items, found {len(SQ_ITEMS)}"

item_inventory = pd.DataFrame([
    {"instrument": k, "sheet": INSTRUMENTS[k]["sheet"], "n_items": len(v),
     "items": ", ".join(v)}
    for k, v in ITEMS.items()
])

display(item_inventory)
item_inventory.to_csv(TABLE_DIR / "Table_02_Item_Inventory.csv", index=False)
print("✓ Item inventory saved")


## 3. Response-code cleaning

The codebook specifies:

- Likert: 1–5
- `99` = Missing/Unclear
- `88` = Not Applicable

For EFA/CFA and quantitative scale calculations, `88` and `99` are treated as missing, not as substantive scores.

We retain the original workbook unchanged.


In [ ]:
# 3. CREATE ANALYSIS COPIES WITH 88/99 RECODED TO NaN

analysis_sheets = {}

for name, df in sheets.items():
    x = df.copy()

    # Recode special missing codes only in numeric survey/assessment items.
    for col in x.columns:
        if re.fullmatch(r"(tq|pq|sq|co|se)\d+", str(col).strip().lower()):
            x[col] = pd.to_numeric(x[col], errors="coerce")
            x.loc[x[col].isin([88, 99]), col] = np.nan

    analysis_sheets[name] = x

print("✓ Analysis copies created.")
print("Original sheets remain untouched.")


In [ ]:
# 3A. SAVE SPECIAL-CODE AUDIT

special_rows = []

for name, df in sheets.items():
    for col in df.columns:
        if re.fullmatch(r"(tq|pq|sq|co|se)\d+", str(col).strip().lower()):
            s = pd.to_numeric(df[col], errors="coerce")
            special_rows.append({
                "sheet": name,
                "item": col,
                "code_88_n": int((s == 88).sum()),
                "code_99_n": int((s == 99).sum()),
                "valid_1_5_n": int(s.isin([1,2,3,4,5]).sum()),
                "other_numeric_n": int((s.notna() & ~s.isin([1,2,3,4,5,88,99])).sum())
            })

special_audit = pd.DataFrame(special_rows)
display(special_audit)

special_audit.to_csv(TABLE_DIR / "Table_03_Response_Code_Audit.csv", index=False)
print("✓ Saved response-code audit")


# 4. Demographic analysis

Demographic variables are analysed separately from measurement items.

For categorical variables: **n and %**.

For numeric/ordered variables where appropriate: **n, mean, SD, median, IQR, minimum, maximum**.

No demographic variable enters EFA.


In [ ]:
# 4. DEMOGRAPHIC FIELD INVENTORY

DEMOGRAPHIC_CANDIDATES = {
    "Teacher": [
        "gender", "age_group", "qualification", "experience",
        "school_type", "class_taught", "location", "division", "district", "upazila"
    ],
    "Parent": [
        "parent_gender", "age_optional", "occupation_optional",
        "education_optional", "number_of_children", "child_grade",
        "location", "division", "district", "upazila"
    ],
    "Student": [
        "grade", "gender", "location", "division", "district", "upazila"
    ],
    "School": [
        "school_type", "location", "division", "district", "upazila"
    ]
}

for label, cols in DEMOGRAPHIC_CANDIDATES.items():
    sheet = {
        "Teacher": "Teacher_Survey",
        "Parent": "Parent_Survey",
        "Student": "Student_Questionnaire",
        "School": "School_Master"
    }[label]

    available = [c for c in cols if c in analysis_sheets[sheet].columns]
    print(f"\n{label}: {available}")


In [ ]:
# 4A. AUTOMATIC DEMOGRAPHIC TABLES

def categorical_summary(df, col):
    x = df[col].dropna()
    out = x.value_counts(dropna=False).rename_axis(col).reset_index(name="n")
    out["percent"] = out["n"] / len(x) * 100
    return out

demographic_outputs = {}

for label, cols in DEMOGRAPHIC_CANDIDATES.items():
    sheet = {
        "Teacher": "Teacher_Survey",
        "Parent": "Parent_Survey",
        "Student": "Student_Questionnaire",
        "School": "School_Master"
    }[label]

    df = analysis_sheets[sheet]
    for col in cols:
        if col in df.columns:
            out = categorical_summary(df, col)
            demographic_outputs[f"{label}_{col}"] = out
            display(out)
            safe = re.sub(r"[^A-Za-z0-9_]+", "_", f"{label}_{col}")
            out.to_csv(TABLE_DIR / f"Demographic_{safe}.csv", index=False)

print("✓ Demographic tables saved")


# 5. Item-level diagnostics

Before EFA, inspect:

- missingness
- mean and SD
- median
- range
- skewness/kurtosis
- floor/ceiling concentration
- variance

The purpose is **diagnosis**, not automatic deletion.


In [ ]:
# 5. ITEM DIAGNOSTICS

def item_diagnostics(df, item_cols):
    rows = []
    for c in item_cols:
        raw = df[c]
        x = pd.to_numeric(raw, errors="coerce").dropna()

        rows.append({
            "item": c,
            "n_valid": len(x),
            "missing_pct": raw.isna().mean() * 100,
            "mean": x.mean(),
            "sd": x.std(ddof=1),
            "median": x.median(),
            "min": x.min(),
            "max": x.max(),
            "skewness": stats.skew(x, bias=False) if len(x) > 2 else np.nan,
            "kurtosis": stats.kurtosis(x, bias=False) if len(x) > 3 else np.nan,
            "floor_pct": (x == 1).mean() * 100,
            "ceiling_pct": (x == 5).mean() * 100,
            "variance": x.var(ddof=1)
        })

    return pd.DataFrame(rows)

diagnostics = {}

for label in ["Teacher", "Parent", "Student"]:
    sheet = INSTRUMENTS[label]["sheet"]
    df = analysis_sheets[sheet]
    d = item_diagnostics(df, ITEMS[label])
    diagnostics[label] = d

    print("\n" + "="*80)
    print(label, "ITEM DIAGNOSTICS")
    print("="*80)
    display(d)

    d.to_csv(TABLE_DIR / f"Table_Item_Diagnostics_{label}.csv", index=False)

print("✓ Item diagnostics saved for all three respondent instruments.")


# 6. EFA — Factorability

### EFA specification

**Teacher, Parent, and Student instruments separately**

- Principal Axis Factoring
- Oblimin rotation
- KMO
- Bartlett's test
- Parallel analysis
- Loading inspection
- Communalities
- Cross-loading review

The number of factors is **not hard-coded**. Parallel analysis provides the empirical starting point, followed by theoretical interpretation against the six QPEI dimensions.


In [ ]:
# 6A. KMO + BARTLETT — FIXED AND SELF-CONTAINED

def prepare_efa_matrix(df, items):
    X = df[items].apply(pd.to_numeric, errors="coerce").copy()

    # Remove columns with no variance
    keep = X.columns[X.nunique(dropna=True) > 1]
    X = X[keep]

    # Complete cases for the correlation/factorability matrix
    X = X.dropna(axis=0, how="any")

    return X, list(keep)

factorability_results = {}

for label in ["Teacher", "Parent", "Student"]:
    sheet = INSTRUMENTS[label]["sheet"]
    df = analysis_sheets[sheet]

    X, usable_items = prepare_efa_matrix(df, ITEMS[label])

    print("\n" + "="*80)
    print(f"{label} EFA FACTORABILITY")
    print("="*80)
    print("Original items:", len(ITEMS[label]))
    print("Usable items:", len(usable_items))
    print("Complete-case N:", len(X))

    if len(usable_items) < 3:
        print("⚠ Not enough items for EFA.")
        continue

    if len(X) < max(50, 5 * len(usable_items)):
        print("⚠ Sample-to-item ratio is limited; interpret cautiously.")

    kmo_all, kmo_model = calculate_kmo(X)
    bart_chi2, bart_p = calculate_bartlett_sphericity(X)

    result = {
        "instrument": label,
        "n_complete_cases": len(X),
        "n_items": len(usable_items),
        "KMO": float(kmo_model),
        "Bartlett_chi2": float(bart_chi2),
        "Bartlett_p": float(bart_p)
    }

    factorability_results[label] = result
    display(pd.DataFrame([result]))

factorability_table = pd.DataFrame(factorability_results.values())
factorability_table.to_csv(TABLE_DIR / "Table_04_EFA_Factorability.csv", index=False)

print("\n✓ Factorability table saved")


In [ ]:
# 6B. PARALLEL ANALYSIS — OBSERVED VS RANDOM EIGENVALUES

def parallel_analysis(df, items, n_iter=500, random_state=42):
    X, usable_items = prepare_efa_matrix(df, items)

    # Standardized matrix
    Z = (X - X.mean()) / X.std(ddof=1)

    obs_corr = np.corrcoef(Z, rowvar=False)
    observed = np.linalg.eigvalsh(obs_corr)[::-1]

    rng = np.random.default_rng(random_state)
    n, p = Z.shape
    random_eigs = np.zeros((n_iter, p))

    for i in range(n_iter):
        R = rng.normal(size=(n, p))
        random_corr = np.corrcoef(R, rowvar=False)
        random_eigs[i] = np.linalg.eigvalsh(random_corr)[::-1]

    mean_random = random_eigs.mean(axis=0)
    p95_random = np.percentile(random_eigs, 95, axis=0)

    suggested = int(np.sum(observed > mean_random))

    pa = pd.DataFrame({
        "factor": np.arange(1, p+1),
        "observed_eigenvalue": observed,
        "mean_random_eigenvalue": mean_random,
        "random_95pct": p95_random
    })

    return pa, suggested, usable_items

parallel_results = {}

for label in ["Teacher", "Parent", "Student"]:
    sheet = INSTRUMENTS[label]["sheet"]
    pa, suggested, usable = parallel_analysis(
        analysis_sheets[sheet], ITEMS[label]
    )

    parallel_results[label] = {
        "table": pa,
        "suggested_factors": suggested,
        "items": usable
    }

    print("\n" + "="*80)
    print(label)
    print("Parallel-analysis suggested factors:", suggested)
    display(pa)

    pa.to_csv(TABLE_DIR / f"Table_05_Parallel_Analysis_{label}.csv", index=False)

    plt.figure(figsize=(8,5))
    plt.plot(pa["factor"], pa["observed_eigenvalue"], marker="o", label="Observed")
    plt.plot(pa["factor"], pa["mean_random_eigenvalue"], marker="o", label="Mean random")
    plt.plot(pa["factor"], pa["random_95pct"], linestyle="--", label="Random 95th percentile")
    plt.axhline(1, linestyle=":", linewidth=1)
    plt.xlabel("Factor")
    plt.ylabel("Eigenvalue")
    plt.title(f"{label} Parallel Analysis")
    plt.legend()
    plt.tight_layout()

    fig_path = FIGURE_DIR / f"Figure_Parallel_Analysis_{label}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print("✓ Saved:", fig_path)


# 7. EFA extraction

The empirical factor solution is estimated separately for each respondent instrument.

We will inspect:

- standardized pattern loadings
- factor correlations
- communalities
- uniqueness
- cross-loadings

**Do not delete an item solely because its loading is inconvenient.** Item content and the QPEI framework must be considered together.


In [ ]:
# 7. RUN EFA

efa_results = {}

for label in ["Teacher", "Parent", "Student"]:
    if label not in parallel_results:
        continue

    n_factors = parallel_results[label]["suggested_factors"]
    items = parallel_results[label]["items"]
    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]

    if n_factors < 1 or len(items) < 3:
        continue

    X = df[items].apply(pd.to_numeric, errors="coerce").dropna()

    # Safety: cannot extract more factors than defensible
    n_factors = min(n_factors, len(items)-1, max(1, len(X)//20))

    fa = FactorAnalyzer(
        n_factors=n_factors,
        method="principal",
        rotation="oblimin"
    )
    fa.fit(X)

    loadings = pd.DataFrame(
        fa.loadings_,
        index=items,
        columns=[f"Factor_{i+1}" for i in range(n_factors)]
    )

    communalities = pd.DataFrame({
        "item": items,
        "communality": fa.get_communalities(),
        "uniqueness": fa.get_uniquenesses()
    })

    factor_corr = pd.DataFrame(
        fa.phi_,
        index=[f"Factor_{i+1}" for i in range(n_factors)],
        columns=[f"Factor_{i+1}" for i in range(n_factors)]
    ) if fa.phi_ is not None else pd.DataFrame()

    efa_results[label] = {
        "model": fa,
        "loadings": loadings,
        "communalities": communalities,
        "factor_correlations": factor_corr,
        "n_factors": n_factors,
        "n": len(X)
    }

    print("\n" + "="*80)
    print(f"{label} EFA — {n_factors} FACTORS")
    print("="*80)

    display(loadings.round(3))
    display(communalities.round(3))

    print("Factor correlations:")
    display(factor_corr.round(3))

    loadings.to_csv(TABLE_DIR / f"Table_06_EFA_Loadings_{label}.csv")
    communalities.to_csv(TABLE_DIR / f"Table_07_EFA_Communalities_{label}.csv", index=False)
    factor_corr.to_csv(TABLE_DIR / f"Table_08_EFA_Factor_Correlations_{label}.csv")

print("\n✓ EFA tables saved")


In [ ]:
# 7A. EFA LOADING HEATMAPS

for label, res in efa_results.items():
    plt.figure(figsize=(max(7, res["n_factors"]*2), max(8, len(res["loadings"])*0.28)))

    sns.heatmap(
        res["loadings"],
        annot=True,
        fmt=".2f",
        center=0,
        cmap="vlag",
        cbar_kws={"label": "Pattern loading"}
    )

    plt.title(f"{label} EFA Pattern Loadings")
    plt.xlabel("Empirical factor")
    plt.ylabel("Survey item")
    plt.tight_layout()

    fig_path = FIGURE_DIR / f"Figure_EFA_Loadings_{label}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print("✓ Saved:", fig_path)


# 8. Framework crosswalk: empirical factors → QPEI dimensions

This is the critical conceptual step.

The six QPEI dimensions come from the established methodology; they are **not replaced by whatever number of factors EFA happens to produce**.

The crosswalk should use:

1. item wording/content;
2. empirical factor loadings;
3. the established QPEI indicators;
4. source/tool alignment.

The methodology specifies, for example, that D1 includes lesson planning, student-centred teaching, TLM use, support for slow learners and feedback; D2 includes learning-outcome clarity, teacher-guide use, continuous assessment and use of assessment results; D3 includes safety, WASH, light/ventilation, seating, playground and classroom readiness; D4 includes reading, writing, numeracy, participation, confidence and aspiration; D5 includes leadership, supervision, monitoring, parent engagement and data use; D6 includes equal participation, slow-learner support, gender sensitivity and disadvantaged-learner support.

**We do not automatically assign tq/pq/sq items to these domains without reviewing the actual item wording.**


In [ ]:
# 8A. FRAMEWORK CROSSWALK TEMPLATE

QPEI_FRAMEWORK = {
    "D1": {
        "name": "Teacher competence and pedagogical practice",
        "weight": 0.20,
        "indicators": [
            "Lesson planning",
            "Student-centred teaching",
            "Teaching-learning materials",
            "Support for slow learners",
            "Feedback"
        ]
    },
    "D2": {
        "name": "Curriculum implementation and assessment",
        "weight": 0.15,
        "indicators": [
            "Learning outcome clarity",
            "Teacher guide use",
            "Continuous assessment",
            "Use of assessment results"
        ]
    },
    "D3": {
        "name": "Learning environment and infrastructure",
        "weight": 0.15,
        "indicators": [
            "Safety",
            "WASH",
            "Light/ventilation",
            "Seating",
            "Playground",
            "Classroom readiness"
        ]
    },
    "D4": {
        "name": "Student learning outcomes and FLN",
        "weight": 0.20,
        "indicators": [
            "Reading",
            "Writing",
            "Numeracy",
            "Participation",
            "Confidence",
            "Aspiration"
        ]
    },
    "D5": {
        "name": "Governance, management, and community support",
        "weight": 0.15,
        "indicators": [
            "Leadership",
            "Supervision",
            "Monitoring",
            "Parent engagement",
            "Data use"
        ]
    },
    "D6": {
        "name": "Equity and inclusiveness",
        "weight": 0.15,
        "indicators": [
            "Equal participation",
            "Slow learner support",
            "Gender sensitivity",
            "Support for disadvantaged learners"
        ]
    }
}

framework_rows = []
for d, spec in QPEI_FRAMEWORK.items():
    for indicator in spec["indicators"]:
        framework_rows.append({
            "dimension": d,
            "dimension_name": spec["name"],
            "weight": spec["weight"],
            "indicator": indicator
        })

framework_table = pd.DataFrame(framework_rows)
display(framework_table)

framework_table.to_csv(TABLE_DIR / "Table_09_QPEI_Framework_Indicators.csv", index=False)

# Empty mapping table to be completed from the questionnaire wording.
crosswalk = pd.DataFrame({
    "instrument": [],
    "item": [],
    "item_translation": [],
    "empirical_factor": [],
    "qpei_dimension": [],
    "framework_indicator": [],
    "direction": [],
    "retained_for_qpei": [],
    "rationale": []
})

crosswalk.to_csv(TABLE_DIR / "Table_10_Empirical_to_QPEI_Crosswalk_TEMPLATE.csv", index=False)

print("✓ Framework table saved")
print("✓ Crosswalk template saved")


# 9. Reliability

Reliability is calculated **after the empirical factor structure is inspected**.

For each retained factor/subscale we will report:

- Cronbach's alpha
- McDonald's omega where estimable
- corrected item-total correlations
- item-level diagnostics

The QPEI itself remains a multidimensional composite, so reliability of the entire QPEI is not treated as the sole validation criterion.


In [ ]:
# 9. RELIABILITY BY EMPIRICAL FACTOR

reliability_rows = []

for label, res in efa_results.items():
    loadings = res["loadings"]

    for factor in loadings.columns:
        # Primary loading = strongest absolute loading
        primary = loadings[factor].abs()
        other = loadings.drop(columns=[factor]).abs().max(axis=1) if len(loadings.columns) > 1 else 0

        selected = loadings.index[
            (primary >= 0.40) &
            ((primary - other) >= 0.10)
        ].tolist()

        if len(selected) >= 2:
            df = analysis_sheets[INSTRUMENTS[label]["sheet"]][selected]
            alpha = pg.cronbach_alpha(data=df.dropna())[0] if PINGOUIN_OK else np.nan

            reliability_rows.append({
                "instrument": label,
                "factor": factor,
                "n_items": len(selected),
                "items": ", ".join(selected),
                "cronbach_alpha": alpha
            })

reliability_table = pd.DataFrame(reliability_rows)
display(reliability_table)

reliability_table.to_csv(TABLE_DIR / "Table_11_Reliability_Empirical_Factors.csv", index=False)
print("✓ Reliability table saved")


# 10. CFA

CFA is **not automatically forced** onto every EFA solution.

A CFA model is generated only after:

- EFA retention is defensible;
- the factor has interpretable item content;
- the factor has enough indicators;
- the model is estimable given sample size.

Fit indices to report:

- χ² and df
- CFI
- TLI
- RMSEA
- SRMR
- standardized loadings
- CR / AVE where appropriate

No modification-index fishing.


In [ ]:
# 10A. CFA MODEL GENERATOR — RUN ONLY AFTER REVIEWING EFA

def generate_cfa_syntax(label, efa_result, min_loading=.40, min_items=3):
    loadings = efa_result["loadings"]
    lines = []

    for factor in loadings.columns:
        primary = loadings[factor].abs()
        other = loadings.drop(columns=[factor]).abs().max(axis=1) if len(loadings.columns) > 1 else 0

        selected = loadings.index[
            (primary >= min_loading) &
            ((primary - other) >= 0.10)
        ].tolist()

        if len(selected) >= min_items:
            lines.append(f"{factor} =~ " + " + ".join(selected))

    return "\n".join(lines)

cfa_syntax = {}

for label, res in efa_results.items():
    syntax = generate_cfa_syntax(label, res)

    cfa_syntax[label] = syntax

    print("\n" + "="*80)
    print(label, "CFA SYNTAX")
    print("="*80)
    print(syntax if syntax else "No factor met the minimum CFA indicator rule.")

    (LOG_DIR / f"{label}_CFA_model.txt").write_text(
        syntax, encoding="utf-8"
    )

print("\n✓ CFA model syntax files saved")
print("Review these models before fitting.")


In [ ]:
# 10B. CFA FITTING — OPTIONAL / REVIEWED MODELS

cfa_results = {}

for label, syntax in cfa_syntax.items():
    if not syntax.strip():
        continue

    df = analysis_sheets[INSTRUMENTS[label]["sheet"]]
    used_items = sorted(
        set(re.findall(r"\b(?:tq|pq|sq)\d+\b", syntax)),
        key=lambda x: (re.sub(r"\D","",x))
    )

    X = df[used_items].apply(pd.to_numeric, errors="coerce").dropna()

    # Avoid attempting clearly underpowered models
    if len(X) < 5 * len(used_items):
        print(f"{label}: skipped CFA fit because complete-case N is too small for the proposed item set.")
        continue

    try:
        model = Model(syntax)
        model.fit(X)
        fit = calc_stats(model)

        cfa_results[label] = {
            "model": model,
            "fit": fit,
            "n": len(X),
            "items": used_items
        }

        print("\n" + "="*80)
        print(label, "CFA FIT")
        print("="*80)
        display(fit)

        fit.to_csv(TABLE_DIR / f"Table_12_CFA_Fit_{label}.csv")

    except Exception as e:
        print(f"{label} CFA error:", repr(e))

print("✓ CFA stage completed where estimable.")


# 11. School-level aggregation

The study's final quality framework operates at the **school level**.

Before aggregating respondent measures, examine whether aggregation is defensible using:

- ICC(1)
- ICC(2)
- respondent counts per school
- substantive appropriateness

The classroom observation and school-environment tools are treated as school/classroom evidence rather than as ordinary respondent-level EFA datasets.


In [ ]:
# 11. SCHOOL SAMPLE / ID AUDIT

for label, sheet in [
    ("Teacher", "Teacher_Survey"),
    ("Parent", "Parent_Survey"),
    ("Student", "Student_Questionnaire"),
    ("Observation", "Classroom_Observation"),
    ("Environment", "School_Environment")
]:
    df = analysis_sheets[sheet]
    if "school_id" in df.columns:
        counts = df["school_id"].value_counts(dropna=False).rename_axis("school_id").reset_index(name="n")
        print("\n", label)
        display(counts)

        counts.to_csv(TABLE_DIR / f"School_Respondent_Counts_{label}.csv", index=False)


# 12. QPEI scoring

The methodology specifies the conceptual six-domain weights:

**20 / 15 / 15 / 20 / 15 / 15**

The native empirical findings remain on the 1–5 scale. When a composite QPEI is explicitly calculated, use:

**1 → 0, 2 → 25, 3 → 50, 4 → 75, 5 → 100**

Then weight the six dimension scores.

This notebook therefore keeps two separate products:

1. **Evidence tables:** native 1–5 results.
2. **Optional QPEI composite:** normalized 0–100 and weighted by the established framework.


In [ ]:
# 12A. QPEI FRAMEWORK WEIGHTS

QPEI_WEIGHTS = {
    "D1": 0.20,
    "D2": 0.15,
    "D3": 0.15,
    "D4": 0.20,
    "D5": 0.15,
    "D6": 0.15
}

QPEI_NAMES = {
    "D1": "Teacher competence and pedagogical practice",
    "D2": "Curriculum implementation and assessment",
    "D3": "Learning environment and infrastructure",
    "D4": "Student learning outcomes and FLN",
    "D5": "Governance, management, and community support",
    "D6": "Equity and inclusiveness"
}

assert abs(sum(QPEI_WEIGHTS.values()) - 1) < 1e-12

weights_table = pd.DataFrame([
    {"dimension": d, "dimension_name": QPEI_NAMES[d], "weight": w}
    for d, w in QPEI_WEIGHTS.items()
])

display(weights_table)
weights_table.to_csv(TABLE_DIR / "Table_13_QPEI_Dimension_Weights.csv", index=False)


In [ ]:
# 12B. NORMALIZATION FUNCTION

def likert_to_qpei100(x):
    # Established framework conversion: 1=0, 2=25, 3=50, 4=75, 5=100
    x = pd.to_numeric(x, errors="coerce")
    return (x - 1) / 4 * 100

print("✓ QPEI 1–5 → 0–100 conversion defined.")


## 12C. QPEI dimension scoring requires the reviewed crosswalk

Do **not** run a fake automatic QPEI merely by splitting tq/pq/sq items by number.

The final dimension score should be based on the reviewed crosswalk from:

**item wording + EFA + methodology indicators + source/tool**

Once the crosswalk is approved, populate `Table_10_Empirical_to_QPEI_Crosswalk_TEMPLATE.csv` and load it here.


In [ ]:
# 12C. LOAD REVIEWED CROSSWALK IF AVAILABLE

CROSSWALK_PATH = TABLE_DIR / "Table_10_Empirical_to_QPEI_Crosswalk_FINAL.csv"

if CROSSWALK_PATH.exists():
    qpei_crosswalk = pd.read_csv(CROSSWALK_PATH)
    print("FINAL crosswalk loaded.")
    display(qpei_crosswalk)
else:
    print("FINAL crosswalk not yet present.")
    print("This is intentional: item-to-dimension assignment requires substantive review.")


# 13. Robustness and sensitivity

After the primary QPEI is constructed, test whether school conclusions are stable under:

- equal dimension weights
- alternative objective weighting methods where appropriate
- weight perturbation / Monte Carlo analysis
- coverage/complete-case alternatives

Robustness should test the conclusion, not be used to choose a preferred ranking.


In [ ]:
# 13A. RANK AGREEMENT HELPER

from scipy.stats import spearmanr, kendalltau

def rank_agreement(df_scores):
    rows = []
    cols = list(df_scores.columns)

    for i, a in enumerate(cols):
        for b in cols[i+1:]:
            rho, rp = spearmanr(df_scores[a], df_scores[b], nan_policy="omit")
            tau, tp = kendalltau(df_scores[a], df_scores[b], nan_policy="omit")

            rows.append({
                "method_a": a,
                "method_b": b,
                "spearman_rho": rho,
                "spearman_p": rp,
                "kendall_tau": tau,
                "kendall_p": tp
            })

    return pd.DataFrame(rows)

print("✓ Robustness helpers ready.")


# 14. Results registry

The registry is updated during the analysis.

Every result should have:

- table/figure ID
- title
- purpose
- source
- method
- numerical result
- interpretation

This prevents the manuscript Results section from drifting away from the analysis.


In [ ]:
# 14. RESULTS.JSON — INITIALIZE OR PRESERVE EXISTING REGISTRY

def initialize_results_registry():
    if RESULTS_PATH.exists():
        try:
            with open(RESULTS_PATH, "r", encoding="utf-8") as f:
                obj = json.load(f)
            print("Existing results.json loaded and preserved.")
            return obj
        except Exception:
            print("Existing results.json could not be parsed; creating a new registry.")

    obj = {
        "project": "Quality Primary Education Index (QPEI)",
        "data_file": str(DATA_PATH),
        "framework": QPEI_NAMES,
        "weights": QPEI_WEIGHTS,
        "tables": [],
        "figures": [],
        "notes": []
    }

    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

    print("Created:", RESULTS_PATH)
    return obj

results_registry = initialize_results_registry()

print("\nCurrent registry:")
print(json.dumps(results_registry, ensure_ascii=False, indent=2)[:5000])


# 15. OUTPUT MANIFEST

This is deliberately near the end, but earlier stages already save their files. If an error occurs later, simply rerun the failed section and then rerun this cell.


In [ ]:
# 15. PRINT AND DISPLAY EVERYTHING GENERATED SO FAR

def output_manifest(root):
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            rows.append({
                "relative_path": str(p.relative_to(root)),
                "size_kb": round(p.stat().st_size / 1024, 2),
                "type": p.suffix.lower()
            })
    return pd.DataFrame(rows)

manifest = output_manifest(OUTPUT_DIR)

print("="*80)
print("QPEI OUTPUT MANIFEST")
print("="*80)
print("Output directory:", OUTPUT_DIR)
print("Number of generated files:", len(manifest))

display(manifest)

manifest.to_csv(OUTPUT_DIR / "OUTPUT_MANIFEST.csv", index=False)

print("\nFILES ON DRIVE:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" •", p)

print("\n✓ Manifest saved:", OUTPUT_DIR / "OUTPUT_MANIFEST.csv")


# 16. Final checklist

- [ ] Master workbook loads successfully
- [ ] Exact instrument sheets detected
- [ ] `tq1–tq30` detected
- [ ] `pq1–pq20` detected
- [ ] `sq1–sq18` detected
- [ ] `co1–co28` detected
- [ ] `se1–se11` detected
- [ ] 88/99 handled as missing for analysis
- [ ] Demographic tables generated
- [ ] Item diagnostics generated
- [ ] KMO/Bartlett generated
- [ ] Parallel analysis generated and figures saved
- [ ] EFA loadings generated
- [ ] Reliability generated
- [ ] CFA models reviewed before fitting
- [ ] School-level respondent counts generated
- [ ] QPEI framework weights preserved
- [ ] Empirical-to-framework crosswalk reviewed before QPEI scoring
- [ ] Results registry maintained
- [ ] Output manifest confirms files exist in Drive
